# Teacher-Student Distillation Basics

Distill a larger teacher's sarcastic-but-correct response style into a smaller student model.

**Roles**

| Role | Purpose |
|---|---|
| Teacher | Produces candidate answers used as training labels |
| Student | Baseline and fine-tuned model being compared |
| Evaluator | Independently scores factual correctness and sarcasm |

The canonical data path is: preserved questions -> live teacher responses -> evaluator scores ->
filtered examples -> train/validation split -> fine-tuning. Training never switches to a checked-in
generated snapshot after demonstrating live generation.

**Cost guard:** inference and training calls are opt-in. This notebook is shipped unexecuted and does
not submit a paid job.

## 1. Setup and provenance

Copy `.env.template` to `.env`, authenticate with `az login`, and install the centrally pinned `../../../requirements.lock`.
The project endpoint and deployment names are supplied only through environment variables.
`data/baseline.jsonl` and `data/qa.jsonl` are byte-for-byte copies of the committed source datasets;
their migration hashes are recorded in `data/SOURCE_DATA_SHA256SUMS.txt`.

In [ ]:
import json
import os
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv()
ROOT = Path.cwd()
DATA = ROOT / "data"
ARTIFACTS = ROOT / "outputs"
ARTIFACTS.mkdir(exist_ok=True)
TRACK_ROOT = next((path for path in (ROOT, *ROOT.parents) if (path / "stage05_runtime.py").is_file()), None)
if TRACK_ROOT is None:
    raise RuntimeError("Run this notebook from its demo directory inside 05-data-and-distillation.")
sys.path.insert(0, str(TRACK_ROOT))
from stage05_runtime import monitor_fine_tuning_job, upload_file_and_wait

PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
FOUNDRY_RUN_PAID_JOBS = os.environ.get("FOUNDRY_RUN_PAID_JOBS", "false").lower() == "true"
FILE_TIMEOUT_SECONDS = float(os.environ.get("FOUNDRY_FILE_TIMEOUT_SECONDS", "600"))
JOB_TIMEOUT_SECONDS = float(os.environ.get("FOUNDRY_JOB_TIMEOUT_SECONDS", "86400"))
POLL_INTERVAL_SECONDS = float(os.environ.get("FOUNDRY_POLL_INTERVAL_SECONDS", "10"))

credential = DefaultAzureCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
model_client = project.get_openai_client()

def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    return path

print("Configuration loaded. Paid training enabled:", FOUNDRY_RUN_PAID_JOBS)

TEACHER_DEPLOYMENT = os.environ["FOUNDRY_TEACHER_DEPLOYMENT"]
STUDENT_DEPLOYMENT = os.environ["FOUNDRY_STUDENT_DEPLOYMENT"]
EVALUATOR_DEPLOYMENT = os.environ["FOUNDRY_EVALUATOR_DEPLOYMENT"]
STUDENT_MODEL = os.environ.get("FOUNDRY_STUDENT_MODEL", "gpt-4.1-nano-2025-04-14")
FINE_TUNED_DEPLOYMENT = os.environ.get("FOUNDRY_FINE_TUNED_DEPLOYMENT", "")
FOUNDRY_RUN_LIVE_EVALUATION = os.environ.get("FOUNDRY_RUN_LIVE_EVALUATION", "false").lower() == "true"

baseline_gold = read_jsonl(DATA / "baseline.jsonl")
questions = read_jsonl(DATA / "qa.jsonl")
assert baseline_gold and questions
assert all({"question", "answer"} <= row.keys() for row in baseline_gold + questions)
question_pool = list(questions)
random.Random(42).shuffle(question_pool)
holdout_count = max(10, len(question_pool) // 5)
generation_questions = question_pool[:-holdout_count]
held_out = question_pool[-holdout_count:]
print(f"Preserved inputs: {len(baseline_gold)} grader examples, {len(questions)} factual questions")

## 2. Define and calibrate the evaluator

The evaluator is not the teacher. It receives the question, candidate answer, and reference factual
answer, then returns JSON with `factual_score`, `sarcasm_score`, `overall_score`, and `reason`.
Incorrect answers are heavily penalized even when their tone is strongly sarcastic. Before using the
evaluator for selection, inspect its scores on the human-curated examples in `baseline.jsonl`.

**Acceptance contract:** the primary goal is teacher-style sarcasm improvement subject to factuality
non-regression. On the same held-out rows, pass when the fine-tuned student's average sarcasm score is
higher than the base student's and its average factual score does not regress. `overall_score` is a
diagnostic because the evaluator prompt does not define a fixed factuality/style weighting.

**Evaluator-calibration finding (2026-10-04):** factual answers with materially different sarcasm
levels all received overall scores of 9-10 while deployment-level sarcasm averages ranged from 2.75
to 5.42. Treat that ceiling effect as evidence to inspect the factual and sarcasm dimensions separately.

In [ ]:
SYSTEM_PROMPT = "Clippy is a factual chatbot that is also sarcastic."
GRADER_PROMPT = '''You are an independent evaluator. Score the candidate from 1 to 10.
Factual correctness is mandatory: a materially wrong answer must receive overall_score <= 2.
Sarcasm should be clear but not abusive. Return only JSON with numeric factual_score,
sarcasm_score, overall_score and a short reason.

Question: {question}
Reference fact: {reference}
Candidate: {candidate}
'''
GRADER_RESPONSE_FORMAT = {
    "type": "json_schema",
    "name": "teacher_student_evaluation",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "factual_score": {"type": "number", "minimum": 1, "maximum": 10},
            "sarcasm_score": {"type": "number", "minimum": 1, "maximum": 10},
            "overall_score": {"type": "number", "minimum": 1, "maximum": 10},
            "reason": {"type": "string"},
        },
        "required": ["factual_score", "sarcasm_score", "overall_score", "reason"],
        "additionalProperties": False,
    },
}

def parse_json_object(text):
    start, end = text.find("{"), text.rfind("}")
    if start < 0 or end <= start:
        raise ValueError(f"Evaluator did not return JSON: {text[:200]}")
    value = json.loads(text[start:end + 1])
    required = {"factual_score", "sarcasm_score", "overall_score", "reason"}
    if not required <= value.keys():
        raise ValueError(f"Evaluator response is missing fields: {required - value.keys()}")
    for name in ("factual_score", "sarcasm_score", "overall_score"):
        if not 1 <= float(value[name]) <= 10:
            raise ValueError(f"{name} must be between 1 and 10")
    return value

def judge(question, reference, candidate):
    response = model_client.responses.create(
        model=EVALUATOR_DEPLOYMENT,
        input=GRADER_PROMPT.format(question=question, reference=reference, candidate=candidate),
        text={"format": GRADER_RESPONSE_FORMAT},
    )
    return parse_json_object(response.output_text)

print("Evaluator recipe defined. Run a small calibration sample before generation.")

## 3. Generate teacher labels and filter them

The teacher output file created below is the only candidate pool accepted by the filtering and
training cells. Every row records model, prompt version, timestamp, source question, reference fact,
teacher answer, and evaluator decision for provenance and versioning.

In [ ]:
FOUNDRY_RUN_LIVE_GENERATION = os.environ.get("FOUNDRY_RUN_LIVE_GENERATION", "false").lower() == "true"
GENERATION_PATH = ARTIFACTS / "teacher_candidates.jsonl"

if FOUNDRY_RUN_LIVE_GENERATION:
    candidates = []
    for row in generation_questions:
        response = model_client.responses.create(
            model=TEACHER_DEPLOYMENT,
            input=f"{SYSTEM_PROMPT}\n\nUser question: {row['question']}",
        )
        teacher_answer = response.output_text.strip()
        candidates.append({
            "question": row["question"],
            "reference_answer": row["answer"],
            "teacher_answer": teacher_answer,
            "evaluation": judge(row["question"], row["answer"], teacher_answer),
            "teacher_deployment": TEACHER_DEPLOYMENT,
            "prompt_version": "sarcasm-v1",
            "generated_at": datetime.now(timezone.utc).isoformat(),
        })
    write_jsonl(GENERATION_PATH, candidates)
else:
    raise RuntimeError("Set FOUNDRY_RUN_LIVE_GENERATION=true to create the teacher dataset. No stale artifact or snapshot fallback is used.")

if not GENERATION_PATH.exists():
    raise RuntimeError("Live generation did not create teacher_candidates.jsonl in this run.")
candidates = read_jsonl(GENERATION_PATH)
print("Live teacher candidates:", len(candidates))

In [ ]:
CUTOFF = float(os.environ.get("TEACHER_SCORE_CUTOFF", "6"))
accepted = [row for row in candidates if float(row["evaluation"]["overall_score"]) >= CUTOFF]
if len(accepted) < 10:
    raise RuntimeError(f"Only {len(accepted)} teacher examples passed; review prompts/evaluator before training.")

rng = random.Random(42)
rng.shuffle(accepted)
split = max(1, int(0.8 * len(accepted)))
train_rows = [{
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["question"]},
        {"role": "assistant", "content": row["teacher_answer"]},
    ]
} for row in accepted[:split]]
validation_rows = [{
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["question"]},
        {"role": "assistant", "content": row["teacher_answer"]},
    ]
} for row in accepted[split:]]

TRAIN_PATH = write_jsonl(ARTIFACTS / "train.jsonl", train_rows)
VALIDATION_PATH = write_jsonl(ARTIFACTS / "validation.jsonl", validation_rows)
print(f"Exact live output -> {len(train_rows)} train / {len(validation_rows)} validation")

## 4. Baseline, fine-tune, deploy, and evaluate

Evaluate the base student, teacher, and deployed fine-tuned student on the same held-out questions
with the same evaluator. Fine-tuning requires an explicit opt-in. For unattended generation, global
training, deployment creation, and terminal evaluation, run `../live_validate.py` from the demo root.

In [ ]:
def evaluate_deployment(deployment, rows):
    scores = []
    for row in rows:
        response = model_client.responses.create(
            model=deployment,
            input=f"{SYSTEM_PROMPT}\n\nUser question: {row['question']}",
        )
        scores.append(judge(row["question"], row["answer"], response.output_text))
    return {
        "count": len(scores),
        "average": sum(float(item["overall_score"]) for item in scores) / len(scores),
        "scores": scores,
    }

print("Held-out rows:", len(held_out))
if FOUNDRY_RUN_LIVE_EVALUATION:
    comparison = {
        "teacher": evaluate_deployment(TEACHER_DEPLOYMENT, held_out),
        "student_base": evaluate_deployment(STUDENT_DEPLOYMENT, held_out),
    }
    if FINE_TUNED_DEPLOYMENT:
        project.deployments.get(FINE_TUNED_DEPLOYMENT)
        comparison["student_fine_tuned"] = evaluate_deployment(FINE_TUNED_DEPLOYMENT, held_out)
    (ARTIFACTS / "evaluation.json").write_text(json.dumps(comparison, indent=2), encoding="utf-8")
    print({name: result["average"] for name, result in comparison.items()})
else:
    print("Set FOUNDRY_RUN_LIVE_EVALUATION=true to run the held-out comparison.")

In [ ]:
if not FOUNDRY_RUN_PAID_JOBS:
    print("Training skipped. Set FOUNDRY_RUN_PAID_JOBS=true only after reviewing generated artifacts.")
else:
    train_file = upload_file_and_wait(
        model_client, TRAIN_PATH, purpose="fine-tune",
        poll_interval_seconds=POLL_INTERVAL_SECONDS, timeout_seconds=FILE_TIMEOUT_SECONDS,
    )
    validation_file = upload_file_and_wait(
        model_client, VALIDATION_PATH, purpose="fine-tune",
        poll_interval_seconds=POLL_INTERVAL_SECONDS, timeout_seconds=FILE_TIMEOUT_SECONDS,
    )
    job = model_client.fine_tuning.jobs.create(
        model=STUDENT_MODEL,
        training_file=train_file.id,
        validation_file=validation_file.id,
        method={"type": "supervised", "supervised": {"hyperparameters": {"n_epochs": 3}}},
        suffix="teacher-student-basics",
        extra_body={"trainingType": "globalStandard"},
    )
    print("Fine-tuning submission accepted; monitoring training to a terminal state.")
    completed_job = monitor_fine_tuning_job(
        model_client, job.id,
        poll_interval_seconds=POLL_INTERVAL_SECONDS, timeout_seconds=JOB_TIMEOUT_SECONDS,
    )
    print("Training outcome:", completed_job.status)

## 5. Representative existing-run gains

The migrated source run reported an evaluator average of **2.9** for the base student,
**5.8** for the teacher, and **5.8** for the distilled student: an absolute gain of **+2.9**.
These are historical, not guaranteed results.

## Conclusion and next step

Distillation is an SFT data pipeline: define independent roles, calibrate the evaluator, retain only
teacher outputs that pass, and compare on held-out data. Job submission means only that the service
accepted the request; training succeeds only after the monitored job reaches `succeeded`. Deployment
is a separate Foundry action, and evaluation is an additional held-out comparison that runs only when
enabled. For this style task, judge acceptance on sarcasm improvement with factual non-regression and
report overall score only as a diagnostic. Next, replace the style task with a domain behavior whose
quality criteria can be reviewed by
subject-matter experts.

*Variability footnote: model sampling, evaluator judgment, service versions, dataset size, and
fine-tuning capacity can change scores between runs. Record model/deployment versions, prompts,
hashes, timestamps, hyperparameters, and evaluation rows with every experiment.*

> **Variability note:** Actual results may vary by model version, data, configuration, region availability, and service conditions.